# Step 3.h — Physiological check of the presence routes of test 4 (FAO ECOCROP)

Test 4 of 3.g compared the no-threshold route (any MIRCA2000 area = presence) with the cellareafrac route (presence
and cropland background above 0.05 % of the cell land area, traces of the crop left out of its table). Outside the
MIRCA2000 cropland the absence of a crop says nothing about its suitability (forest, tundra, protected areas, no
people), so the control zones (Antarctica, Greenland, Sahara, Changtang) were the only places where the answer was
known. GAEZ v5 is no reference there: it is too strict (crops grown in MIRCA2000 cells unsuitable in GAEZ), so its
"unsuitable" cannot show that the SVM is wrong.

This notebook uses the crop physiology instead: the **absolute minimum temperature** of FAO ECOCROP (`TMIN`, mean
temperature of the growing period below which the crop does not grow; parameters of the FAO ECOCROP database as
distributed by the R package Recocrop, `ecocropPars`; EcoCrop model: Ramirez-Villegas et al. 2013, *Agric. For.
Meteorol.* 170, 67–78). The limits are wide (not a strict model like GAEZ) and do not depend on land use.

| crop (rainfed) | ECOCROP entry | TMIN (°C) | temperature compared |
|---|---|---|---|
| maize | *Zea mays* L. s. mays | 10 | season mean |
| potato | *Solanum tuberosum* L. | 7 | season mean |
| soybean | *Glycine max* (L.) Merrill | 10 | season mean |
| rice | *Oryza sativa* japonica (paddy and upland; indica 16) | 10 | season mean |
| spring wheat | *Triticum aestivum* L. | 5 | season mean |
| winter wheat, wheat | *Triticum aestivum* L. | 5 | phase 2 (last 60 days) |

Season mean temperature of the BIO variables of variant A (the calendar the SVM sees, MIRCA2000 or gap-filled):
phase 1 (sowing → 60 days before maturity) mean (BIO01 + BIO02) / 2, phase 2 (last 60 days) mean (BIO06 + BIO07) / 2,
weighted by their lengths. Winter wheat overwinters (dormancy, vernalisation), so its season mean may be below 5 °C
where it grows well: for winter wheat and wheat (winter + spring) the mean of phase 2 (spring and summer growth, grain
filling) is compared. A season mean below TMIN implies at least one month below TMIN, so the cell is unsuitable for
EcoCrop: the check is lenient, it flags only cells that are clearly too cold.

For each route (models of test 4: variant A, ERA5 radiation, one-pass), the ERA5 suitability probability of 4.g
(0 where novel: MESS < 0 or NT2 > 1) is compared with the check, at the probability 0.5 (binary map of 4.g) and
0.2: where the crop cannot grow, a sound model gives a probability near 0, not only below 0.5. With `BALANCED`
the probability is the one of 4.g, brought to equal class priors (`cu.balanced_proba`): the Platt scaling of
libsvm carries the share of presences of the training table (maize: the majority), which shifts every
probability of that crop up; without it (`BALANCED = False`) the Platt probability (first run of 3 November).
1. **control**: share of the MIRCA2000 harvested area, of the presence cells (> 0.05 %) and of the trace cells
   (> 0, ≤ 0.05 %) below TMIN: about 0 for the real presences if the limits are right;
2. **suitable but too cold** at each probability (Mha and % of the suitable area): on the MIRCA2000 cropland,
   outside it, north of 50° N and on the Tibetan plateau (27–40° N, 75–105° E) outside the cropland, Antarctica;
   and the **mean probability of the cells below TMIN** in each region (no threshold needed);
3. maps: suitable (≥ 0.5), and below TMIN with a probability 0.2–0.5 or ≥ 0.5.

Outputs in `<out_root>/3_svm_train/ERA5_<cellareafrac route>/`: `ecocrop_check_<set>[_balanced]_<...>.csv`,
`ecocrop_check_maps_<set>[_balanced]_<crop>.png`.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt, joblib
from matplotlib.colors import ListedColormap
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb

# ---- parameters
CLIMATE, PERIOD = "era5", ("1981", "2014")       # training climate (tables of 1.1.d)
SET = "A_60d_rad_era5"                            # models of test 4 (3.g)
CELL_AREA_THR = 0.0005                            # threshold of the cellareafrac route
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
THRESHOLDS = [0.2, 0.5]                           # 0.5 = binary suitability of 4.g; 0.2 = a low probability
BALANCED = True                                   # probability at equal class priors, as 4.g (False = Platt)
# FAO ECOCROP absolute minimum temperature (degC) and the temperature compared ("season" or "phase2")
ECOCROP = {"Maize_rf": (10, "season"), "Potato_rf": (7, "season"), "Soybean_rf": (10, "season"), "Rice_rf": (10, "season"),
           "Wheat-spring_rf": (5, "season"), "Wheat-winter_rf": (5, "phase2"), "Wheat_rf": (5, "phase2")}

TAG = f"{CLIMATE.upper()}_{PERIOD[0]}_{PERIOD[1]}_ecearth3"
SUF = "_balanced" if BALANCED else ""
ROUTES = {"no threshold": None, f"cellareafrac {100 * CELL_AREA_THR:g} %": CELL_AREA_THR}
OUT0 = cu.out_dir(CFG, "3_svm_train", f"{CLIMATE.upper()}_{cu.route(CELL_AREA_THR)}")
GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
CRO_ANY = cu.mirca_cropland(CFG).values                 # any MIRCA2000 cropland (no threshold)
LAND_HA = np.nan_to_num(cu.land_area_ha(CFG)) * LAND
LAT2, LON2 = np.meshgrid(GRID["lat"], GRID["lon"], indexing="ij")
REGIONS = {"MIRCA cropland": LAND & CRO_ANY,
           "outside cropland": LAND & ~CRO_ANY,
           "north of 50 N, outside cropland": LAND & ~CRO_ANY & (LAT2 > 50),
           "Tibetan plateau, outside cropland": LAND & ~CRO_ANY & (LAT2 > 27) & (LAT2 < 40) & (LON2 > 75) & (LON2 < 105),
           "Antarctica": LAND & (LAT2 < -60)}
print({r: int(m.sum()) for r, m in REGIONS.items()}, "land cells per region | outputs", OUT0)

## 1. Temperature of the growing period, suitability of each route

In [ ]:
def growing_temperature(ds, which):
    # mean temperature of the season (phases weighted by their lengths) or of phase 2, from the BIO file of 1.1.d
    t1 = (ds["BIO01"].values + ds["BIO02"].values) / 2
    t2 = (ds["BIO06"].values + ds["BIO07"].values) / 2
    if which == "phase2":
        return t2
    sow, ph2, mat = (ds[v].values for v in ("sowing_doy", "phase2_start_doy", "maturity_doy"))
    l1 = np.mod(ph2 - sow, 365)
    l2 = np.mod(mat - ph2, 365) + 1
    return np.where(l1 > 0, (l1 * t1 + l2 * t2) / (l1 + l2), t2)


def suitability(rname, crop):
    # ERA5 suitability probability of 4.g with the model of the route: 0 where novel (MESS < 0 or NT2 > 1), NaN off land
    bdir = cu.out_dir(CFG, "1_biovariables", rname, "A_60d")
    b = joblib.load(cu.out_dir(CFG, "3_svm_train", rname, SET) / f"svm_model_{TAG}_{crop}.joblib")
    df = pd.read_csv(bdir / f"training_dataset_{TAG}_{crop}.csv", usecols=["lat", "lon"])
    ds = xr.open_dataset(bdir / f"BIOvariables_{TAG}_{crop}.nc").load()
    B = np.stack([ds[c].values for c in b["bio_cols"]], axis=-1)
    ii = np.searchsorted(GRID["lat"], df.lat.values - 1e-6); jj = np.searchsorted(GRID["lon"], df.lon.values - 1e-6)
    ref = B[ii, jj]; ref = ref[np.isfinite(ref).all(axis=1)]
    ok = np.isfinite(B).all(axis=-1) & LAND
    p, m, n = (np.full(LAND.shape, np.nan) for _ in range(3))
    p[ok] = b["model"].predict_proba(b["scaler"].transform(B[ok]))[:, 1]
    if BALANCED:
        p[ok] = cu.balanced_proba(p[ok], b["n_class1"], b["n_class0"])
    m[ok] = cu.mess(B[ok], ref)[0]
    n[ok] = cu.nt2(B[ok], ref)
    return np.where(ok, np.where((m >= 0) & (n <= 1), p, 0.0), np.nan), ds


rows, maps = [], {}
for crop in CROPS:
    tmin, which = ECOCROP[crop]
    area = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)    # MIRCA2000 harvested area, traces included
    frac = cu.cell_area_frac(CFG, area)
    pres, trace = LAND & (frac > CELL_AREA_THR), LAND & (area > 0) & (frac <= CELL_AREA_THR)
    maps[crop] = {}
    for name, thr in ROUTES.items():
        q, ds = suitability(f"{CLIMATE.upper()}_{cu.route(thr)}", crop)
        T = growing_temperature(ds, which)
        cold = LAND & np.isfinite(T) & (T < tmin)
        r = {"crop": crop, "route": name, "ECOCROP TMIN (degC)": tmin, "temperature": which,
             "MIRCA area below TMIN (%)": 100 * area[cold].sum() / area.sum(),
             "presence cells below TMIN (%)": 100 * (pres & cold).sum() / pres.sum(),
             "trace cells below TMIN (%)": 100 * (trace & cold).sum() / max(trace.sum(), 1),
             "trace cells": int(trace.sum())}
        for reg, rm in REGIONS.items():
            r[f"{reg}: mean probability below TMIN"] = float(np.nanmean(q[cold & rm])) if (cold & rm).any() else np.nan
            for pt in THRESHOLDS:
                suit = np.nan_to_num(q) >= pt
                s = LAND_HA[suit & rm].sum() / 1e6
                c = LAND_HA[suit & cold & rm].sum() / 1e6
                r[f"{reg}: suitable (Mha), p >= {pt}"] = s
                r[f"{reg}: suitable below TMIN (Mha), p >= {pt}"] = c
                r[f"{reg}: suitable below TMIN (% of suitable), p >= {pt}"] = 100 * c / s if s else np.nan
        rows.append(r)
        maps[crop][name] = (np.nan_to_num(q), cold, trace & cold)
    print(f"{crop}: done", flush=True)

## 2. Results

In [ ]:
chk = pd.DataFrame(rows).set_index(["crop", "route"])
chk.to_csv(OUT0 / f"ecocrop_check_{SET}{SUF}_{TAG}.csv")
with pd.option_context("display.max_rows", 200, "display.max_columns", 40, "display.width", 250):
    print("(1) control: MIRCA2000 presences and traces below the ECOCROP TMIN (the routes share the MIRCA data)")
    display(chk.xs(list(ROUTES)[0], level="route")[["ECOCROP TMIN (degC)", "temperature", "MIRCA area below TMIN (%)",
                                                    "presence cells below TMIN (%)", "trace cells", "trace cells below TMIN (%)"]].round(2))
    print("(2) mean probability of the cells below TMIN (0 where novel)")
    display(chk[[f"{reg}: mean probability below TMIN" for reg in REGIONS]].rename(columns=lambda c: c.split(":")[0]).round(3))
    for pt in THRESHOLDS:
        for k in ("suitable (Mha)", "suitable below TMIN (Mha)", "suitable below TMIN (% of suitable)"):
            print(f"(2) {k}, probability >= {pt}")
            display(chk[[f"{reg}: {k}, p >= {pt}" for reg in REGIONS]].rename(columns=lambda c: c.split(":")[0]).round(1))

In [ ]:
names = list(ROUTES)
cmap = ListedColormap(["#eeeeee", "#9acd32", "#fdae61", "#d7301f"])   # not suitable | suitable (>= 0.5) | below TMIN, 0.2-0.5 | below TMIN, >= 0.5
lo, hi = THRESHOLDS
for crop in CROPS:
    fig, axes = plt.subplots(1, 2, figsize=(18, 4.8))
    for ax, name in zip(axes, names):
        q, cold, tc = maps[crop][name]
        z = np.where(cold & (q >= hi), 3, np.where(cold & (q >= lo), 2, np.where(q >= hi, 1, 0)))
        ax.pcolormesh(GRID["lon"], GRID["lat"], np.where(LAND, z, np.nan), cmap=cmap, vmin=-0.5, vmax=3.5, shading="nearest")
        ii, jj = np.nonzero(tc)
        ax.scatter(GRID["lon"][jj], GRID["lat"][ii], s=2, c="k", label="MIRCA trace below TMIN")
        c1, c2 = LAND_HA[cold & (q >= hi)].sum() / 1e6, LAND_HA[cold & (q >= lo)].sum() / 1e6
        ax.set_title(f"{crop}, {name}: green p >= {hi}; below ECOCROP TMIN: red p >= {hi} ({c1:.0f} Mha), "
                     f"orange {lo}-{hi} (total >= {lo}: {c2:.0f} Mha)", fontsize=9)
        ax.set_aspect("equal"); ax.legend(loc="lower left", fontsize=7)
    plt.tight_layout(); plt.savefig(OUT0 / f"ecocrop_check_maps_{SET}{SUF}_{crop}.png", dpi=80); plt.show()